# Student Performance Prediction using Python, MongoDB, and Scikit-Learn
**A Beginner-Friendly Machine Learning Project for B.Tech CSE Students**

### Project Highlights:
- **Database:** MongoDB (`student_performance_db`)
- **Connection Library:** PyMongo
- **Dataset Size:** 1,000 Student Records
- **ML Algorithm:** Linear Regression
- **Evaluation Metrics:** $R^2$ Score and Mean Absolute Error (MAE)

## 1. Install & Import Required Libraries

In [ ]:
# Import all required packages
import numpy as np
import pandas as pd
from pymongo import MongoClient
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

## 2. Generate 1,000 Realistic Student Records & Store in MongoDB

In [ ]:
# Set random seed for consistency
np.random.seed(42)
num_records = 1000

# 1. Features generation
student_ids = [f"STU{1001 + i}" for i in range(num_records)]
study_hours = np.round(np.random.uniform(1.0, 9.5, num_records), 1)
attendance = np.round(np.random.uniform(60.0, 99.0, num_records), 1)
previous_marks = np.round(np.random.uniform(40.0, 95.0, num_records), 1)
assignments_score = np.random.randint(4, 11, num_records)

# 2. Calculate realistic Final Marks with realistic noise
noise = np.random.normal(0, 2.5, num_records)
final_marks = (0.45 * previous_marks + 3.5 * study_hours + 0.25 * attendance + 1.5 * assignments_score + noise)
final_marks = np.round(np.clip(final_marks, 30.0, 100.0), 1)

# Combine into DataFrame
df_raw = pd.DataFrame({
    "student_id": student_ids,
    "study_hours": study_hours,
    "attendance": attendance,
    "previous_marks": previous_marks,
    "assignments_score": assignments_score,
    "final_marks": final_marks
})

# Connect to MongoDB and save records
client = MongoClient("mongodb://localhost:27017/")
db = client["student_performance_db"]
collection = db["students"]

collection.delete_many({})
collection.insert_many(df_raw.to_dict(orient="records"))
print(f"Successfully stored {collection.count_documents({})} documents in MongoDB!")

## 3. Fetch Data from MongoDB using PyMongo & Show Sample Records

In [ ]:
# Fetch all documents from MongoDB
data_from_db = list(collection.find())
df = pd.DataFrame(data_from_db)

print(f"Fetched {len(df)} records from MongoDB.")
# Display 5 sample records
df.head(5)

## 4. Data Preprocessing & Train-Test Split (80% / 20%)

In [ ]:
# Drop MongoDB '_id' and identifier 'student_id'
clean_df = df.drop(columns=['_id', 'student_id'])

# Check for missing values
print("Missing values check:")
print(clean_df.isnull().sum())

# Separate Features (X) and Target (y)
X = clean_df[['study_hours', 'attendance', 'previous_marks', 'assignments_score']]
y = clean_df['final_marks']

# Split: 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set: {X_train.shape[0]} rows, Testing set: {X_test.shape[0]} rows")

## 5. Train Linear Regression Model

In [ ]:
# Initialize and fit model
model = LinearRegression()
model.fit(X_train, y_train)

# Print learned weights
for feature, weight in zip(X.columns, model.coef_):
    print(f"Weight for {feature:20s}: {weight:.4f}")
print(f"Intercept (c): {model.intercept_:.4f}")

## 6. Evaluation Metrics & Comparison Table

In [ ]:
# Predict on testing dataset
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R-squared (R2) Score : {r2:.4f} (Accuracy: {r2*100:.2f}%)")
print(f"Mean Absolute Error  : {mae:.2f} marks")
print(f"Root Mean Sq Error   : {rmse:.2f} marks")

# Display 10 Sample Predictions
comparison = pd.DataFrame({
    "Actual Marks": y_test.values[:10],
    "Predicted Marks": np.round(y_pred[:10], 1),
    "Absolute Error": np.round(np.abs(y_test.values[:10] - y_pred[:10]), 1)
})
comparison

## 7. Visualizing Actual vs Predicted Marks

In [ ]:
plt.figure(figsize=(12, 5))

# Scatter plot
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, color='#2563EB', alpha=0.6, edgecolors='k', s=40)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='#DC2626', linestyle='--', linewidth=2, label='Perfect Fit')
plt.title("Actual vs Predicted Marks", fontweight='bold')
plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

# Bar chart for 10 samples
plt.subplot(1, 2, 2)
indices = np.arange(10)
width = 0.35
plt.bar(indices - width/2, y_test.values[:10], width, label='Actual', color='#3B82F6')
plt.bar(indices + width/2, y_pred[:10], width, label='Predicted', color='#10B981')
plt.title("Comparison for 10 Students", fontweight='bold')
plt.xlabel("Student Index")
plt.ylabel("Marks")
plt.xticks(indices, [f"S{i+1}" for i in indices])
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()

## 8. Test Prediction with a New Custom Student

In [ ]:
custom_student = pd.DataFrame([{
    'study_hours': 7.0,
    'attendance': 90.0,
    'previous_marks': 82.0,
    'assignments_score': 9
}])

predicted_score = model.predict(custom_student)[0]
print(f"Predicted Final Marks for custom student: {predicted_score:.1f} / 100")